# CompuCell3D through process-bigraph — a capability demonstration

_Investigation `cpm-demonstrations` — coder reproduction notebook._

**Question.** What multicellular behaviours does the real CompuCell3D engine expose once it is
wrapped as a single process-bigraph Process, and can each be driven, observed
through typed ports, and rendered as a shareable artifact without leaving the
bigraph framework?

Four self-contained CompuCell3D demonstrations, each driven through the
`CompuCell3DProcess` wrapper on the real cc3d 4.6.0 engine and rendered as
committed artifacts (an animated lattice scene + static charts + a metrics CSV):

  1. **Differential-adhesion cell sorting** — two cell types with unequal
     contact energies reorganise (Steinberg sorting).
  2. **Chemotaxis** — responder cells migrate on a secreted, diffusing signal
     field.
  3. **Growth & division** — cells grow to a target volume and undergo mitosis.
  4. **Spheroid invasion** — a proliferating core expands with an invasive front.

---

**How to use this notebook**

1. **Run ▸ Run All Cells** to reproduce every study end to end.
2. To explore, edit a study's **Parameters** cell, then re-run from its **Run** section — the figures regenerate with your values.
3. The **Setup** cell at the top is collapsed scaffolding (environment + render helpers); expand it only to inspect. Set `RERUN = False` there to render the committed `runs.db` without re-simulating.

The prose states the **question and parameters**; the figures each run produces are the results.


In [ ]:
"""Setup — environment & helpers (runs automatically; open only to inspect).

Generated by vivarium-workbench (notebook_export). This cell resolves the repo,
registers the workspace Core, and defines the render helpers; the science lives
in the sections below. Each study is re-run live with the workspace's own
process-bigraph protocol and its figures rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-compucell3d/viva-compucell3d').is_dir():
    REPO = Path('/home/runner/work/viva-compucell3d/viva-compucell3d')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# Keep the output clean: composite discovery and the pydantic models emit
# benign UserWarnings (optional packages it skips, a field-name shadow) and the
# simulators emit DeprecationWarnings — none are actionable from a reproduction
# notebook, so silence those two categories. Remove this to see them.
import warnings as _warnings
_warnings.filterwarnings("ignore", category=UserWarning)
_warnings.filterwarnings("ignore", category=DeprecationWarning)

# --- standard process-bigraph protocol: register the workspace's Core ---
from pbg_compucell3d.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

def materialize_composite(spec_id, overrides=None):
    """Build a composite into an editable payload {name, description,
    parameters, state, _overrides, _id}. Works for @composite_generator
    functions via the process-bigraph registry (the common case); ``state``
    is the built document (process nodes at top level)."""
    import importlib
    from process_bigraph.composite_generator import (
        _REGISTRY, apply_core_extensions, build_generator)
    try:
        from process_bigraph.composite_generator import discover_generators
        discover_generators()
    except Exception:
        pass
    entry = _REGISTRY.get(spec_id)
    if entry is None:
        # editable installs are invisible to packages_distributions, so
        # discover alone can miss them — prime the module the id names.
        try:
            importlib.import_module(spec_id.rsplit('.', 1)[0])
            from process_bigraph.composite_generator import discover_generators as _dg
            _dg()
        except Exception:
            pass
        entry = _REGISTRY.get(spec_id)
    if entry is None:
        raise KeyError('composite not registered: ' + repr(spec_id))
    params = dict(entry.parameters or {})
    ov = {k: (v.get('default') if isinstance(v, dict) else None)
          for k, v in params.items()}
    if overrides:
        ov.update({k: val for k, val in overrides.items() if k in params})
    _c = apply_core_extensions(entry, core)
    try:
        doc = build_generator(entry, ov or None, core=_c)
    except TypeError:
        doc = entry.func(_c, **(ov or {}))
    return {'name': entry.name, 'description': entry.description,
            'parameters': params, 'state': doc, '_overrides': ov, '_id': spec_id}

import base64 as _b64, pathlib as _pl
_STATIC_VIZ_SCHEMES = ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:')
def _render_viz_process(address, config):
    """Render a Visualization process/step (e.g. ``local:<Class>``) to
    HTML: wrap it as a one-step composite and run it once — the viz writes
    Plotly HTML to its ``html`` output port. Mirrors the workbench's own
    env-worker viz path. Self-running vizes need no run data."""
    from process_bigraph import Composite
    _doc = {
        'inputs_store': {}, 'output_store': '',
        'visualization': {
            '_type': 'step', 'address': address,
            'config': {k: v for k, v in (config or {}).items()
                       if k not in ('inputs_map', 'sources')},
            'inputs': {}, 'outputs': {'html': ['output_store']},
        },
    }
    _comp = Composite({'state': _doc}, core=core)
    _comp.run(1)
    _html = _comp.state.get('output_store')
    if isinstance(_html, dict):
        _html = _html.get('value') or _html.get('_value') or ''
    return _html if isinstance(_html, str) else ''
def _render_one(address, config, runs_db, study_yaml, overrides=None):
    """Generic figure renderer (no workspace render_study_viz.py).
    A static ``image:``/``file:`` (or bare-path) address embeds the file;
    any other address (``local:<VizProcess>``) is run as a Visualization
    and its HTML returned. ``overrides`` (the composite's edited parameter
    dict) is merged into the viz config first so a self-running figure
    reflects edited parameters; a viz that does not accept those keys
    falls back to the base config."""
    addr = str(address or '')
    _head = addr.split('/', 1)[0]
    _is_static = (not addr) or (':' not in _head) or addr.startswith(_STATIC_VIZ_SCHEMES)
    if not _is_static:
        if core is None:
            return f'<p style="color:#b91c1c">no core to render {address}</p>'
        _base = {k: v for k, v in (config or {}).items()}
        _tries = ([{**_base, **overrides}] if overrides else []) + [_base]
        _err = ''
        for _cfg in _tries:
            try:
                _h = _render_viz_process(addr, _cfg)
            except Exception as _e:
                _err = (f'<p style="color:#b91c1c">visualization {address} not renderable yet: '
                        f'{type(_e).__name__}: {_e}</p>')
                continue
            return _h or f'<p style="color:#6b7280">{address} produced no HTML</p>'
        return _err
    for _scheme in _STATIC_VIZ_SCHEMES:
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: Differential-adhesion cell sorting (`cell-sorting`)

**Question.** Given two cell types with unequal contact energies, does the wrapped CompuCell3D
engine reorganise an initially mixed blob so that like cells associate — the
Steinberg differential-adhesion effect — driven entirely through the Process's
typed config?

**Objective.** Initialise a mixed two-type blob on an 80x80 lattice, advance the real engine for
3000 Monte-Carlo steps through `CompuCell3DProcess`, and read cell counts and
morphometry back through the wrapper's output ports.

**Hypothesis.** With type-A homotypic adhesion strong (low contact energy) and type-B weaker,
minimising interfacial energy drives the two populations to segregate while the
total cell count is conserved.

**Claim.** The wrapper drives genuine differential-adhesion sorting: the 45-cell population
(16 type-A, 29 type-B) is conserved while the mean cell surface rises from 22.3 to
28.0 px over 3000 MCS as cells reorganise and like-cell boundaries consolidate.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `sorting` | `pbg_compucell3d.processes` | 0 | dim_x=80, dim_y=80, total_mcs=3000 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**Composite `pbg_compucell3d.processes`** — `spec_pbg_compucell3d_processes` (materialized from its `@composite_generator`)


In [ ]:
spec_pbg_compucell3d_processes = materialize_composite('pbg_compucell3d.processes')
describe_spec(spec_pbg_compucell3d_processes)

In [ ]:
# === Edit parameters for this composite, then run the Run cell ===
# ov_pbg_compucell3d_processes maps each tunable parameter to its current value (the generator's
# defaults). Edit any entry; the Run cell rebuilds the composite with it.
ov_pbg_compucell3d_processes = dict(spec_pbg_compucell3d_processes['_overrides'])
import json as _json; print(_json.dumps(ov_pbg_compucell3d_processes, indent=2, default=str))
# e.g.:  ov_pbg_compucell3d_processes['<parameter>'] = <value>

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: cell-sorting ===
STUDY = 'cell-sorting'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_sorting = 100
INTERVAL_sorting = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_pbg_compucell3d_processes if 'ov_pbg_compucell3d_processes' in dir() else None  # edits from the panel cell, if any
            spec_pbg_compucell3d_processes = materialize_composite('pbg_compucell3d.processes', _ov)
            _comp = Composite({'state': spec_pbg_compucell3d_processes['state']}, core=core)
            _comp.run(STEPS_sorting)
        print('ran', 'sorting', f'(pbg_compucell3d.processes) for', STEPS_sorting, 'steps')
    except Exception as _e:
        print(f"sorting not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Population, composition & morphometry**


In [ ]:
# Population, composition & morphometry
show_viz(_render_one('charts/', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| Differential adhesion drives sorting on the real engine _(demonstration)_ |  |  |


## Study: Chemotaxis on a secreted, diffusing field (`chemotaxis`)

**Question.** Can the wrapper couple a secretion source, a finite-difference diffusion field, and
a chemotaxis response inside one CompuCell3D Process so that responder cells migrate
directionally on the resulting gradient?

**Objective.** Run the secretion + diffusion + chemotaxis composite on an 80x80 lattice for 2000
Monte-Carlo steps through `CompuCell3DProcess`, exposing the concentration field in
the animated scene and reading population back through the output ports.

**Hypothesis.** Source cells secreting a diffusible signal set up a spatial gradient; responder cells
with a chemotaxis term bias their motion up that gradient and migrate toward the
source region.

**Claim.** The composite runs the real engine with an emergent, evolving concentration field and
drives net responder migration toward the source. Under this configuration the
responder population contracts over the run (45 -> 16 cells across 2000 MCS) as cells
move up-gradient — reported as observed, not as a tuned recruitment yield.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `chemotaxis` | `pbg_compucell3d.processes` | 0 | dim_x=80, dim_y=80, total_mcs=2000 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**Composite `pbg_compucell3d.processes`** — `spec_pbg_compucell3d_processes` (materialized from its `@composite_generator`)


In [ ]:
spec_pbg_compucell3d_processes = materialize_composite('pbg_compucell3d.processes')
describe_spec(spec_pbg_compucell3d_processes)

In [ ]:
# === Edit parameters for this composite, then run the Run cell ===
# ov_pbg_compucell3d_processes maps each tunable parameter to its current value (the generator's
# defaults). Edit any entry; the Run cell rebuilds the composite with it.
ov_pbg_compucell3d_processes = dict(spec_pbg_compucell3d_processes['_overrides'])
import json as _json; print(_json.dumps(ov_pbg_compucell3d_processes, indent=2, default=str))
# e.g.:  ov_pbg_compucell3d_processes['<parameter>'] = <value>

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: chemotaxis ===
STUDY = 'chemotaxis'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_chemotaxis = 100
INTERVAL_chemotaxis = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_pbg_compucell3d_processes if 'ov_pbg_compucell3d_processes' in dir() else None  # edits from the panel cell, if any
            spec_pbg_compucell3d_processes = materialize_composite('pbg_compucell3d.processes', _ov)
            _comp = Composite({'state': spec_pbg_compucell3d_processes['state']}, core=core)
            _comp.run(STEPS_chemotaxis)
        print('ran', 'chemotaxis', f'(pbg_compucell3d.processes) for', STEPS_chemotaxis, 'steps')
    except Exception as _e:
        print(f"chemotaxis not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Population, composition & morphometry**


In [ ]:
# Population, composition & morphometry
show_viz(_render_one('charts/', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| Secretion + diffusion + chemotaxis runs and drives migration _(demonstration)_ |  |  |


## Study: Growth & division (mitosis) (`growth-division`)

**Question.** Can the wrapper drive volume growth and mitotic division through CompuCell3D so that a
small seed population expands into a large one, read back through the output ports?

**Objective.** Seed 25 cells on a 150x150 lattice, advance 3000 Monte-Carlo steps through
`CompuCell3DProcess` with growth + mitosis enabled, and track population and mean
volume through the output ports.

**Hypothesis.** Cells whose target volume increases each Monte-Carlo step grow, and once they reach a
division threshold they split via the CPM mitosis steppable — so the population grows
roughly geometrically while mean cell volume stays near the post-division target.

**Claim.** The wrapper drives real CPM growth and mitosis: the population expands from 25 to 800
cells over 3000 MCS (15 -> 480 type-A, 10 -> 320 type-B) while mean cell volume rises
modestly from 24.8 to 28.1 px, consistent with divide-at-threshold dynamics.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `growth` | `pbg_compucell3d.processes` | 0 | dim_x=150, dim_y=150, total_mcs=3000 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**Composite `pbg_compucell3d.processes`** — `spec_pbg_compucell3d_processes` (materialized from its `@composite_generator`)


In [ ]:
spec_pbg_compucell3d_processes = materialize_composite('pbg_compucell3d.processes')
describe_spec(spec_pbg_compucell3d_processes)

In [ ]:
# === Edit parameters for this composite, then run the Run cell ===
# ov_pbg_compucell3d_processes maps each tunable parameter to its current value (the generator's
# defaults). Edit any entry; the Run cell rebuilds the composite with it.
ov_pbg_compucell3d_processes = dict(spec_pbg_compucell3d_processes['_overrides'])
import json as _json; print(_json.dumps(ov_pbg_compucell3d_processes, indent=2, default=str))
# e.g.:  ov_pbg_compucell3d_processes['<parameter>'] = <value>

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: growth-division ===
STUDY = 'growth-division'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_growth = 100
INTERVAL_growth = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_pbg_compucell3d_processes if 'ov_pbg_compucell3d_processes' in dir() else None  # edits from the panel cell, if any
            spec_pbg_compucell3d_processes = materialize_composite('pbg_compucell3d.processes', _ov)
            _comp = Composite({'state': spec_pbg_compucell3d_processes['state']}, core=core)
            _comp.run(STEPS_growth)
        print('ran', 'growth', f'(pbg_compucell3d.processes) for', STEPS_growth, 'steps')
    except Exception as _e:
        print(f"growth not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Population, composition & morphometry**


In [ ]:
# Population, composition & morphometry
show_viz(_render_one('charts/', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| Growth to target volume and mitotic division on the real engine _(demonstration)_ |  |  |


## Study: Tumor spheroid invasion (`spheroid-invasion`)

**Question.** Can the wrapper reproduce an invasive-growth morphology in CompuCell3D — a
proliferating core that expands with a rough, fingering front rather than a smooth
disc — driven through the Process config?

**Objective.** Seed 25 cells on a 150x150 lattice, advance 4000 Monte-Carlo steps through
`CompuCell3DProcess`, and track population and mean surface through the output ports.

**Hypothesis.** A proliferating cell population with adhesion and surface parameters that favour an
extended interface expands as an invasive front: cell count grows and mean cell
surface rises faster than for compact growth, reflecting a rougher boundary.

**Claim.** The wrapper reproduces an invasive-growth morphology on the real engine: the
population expands from 25 to 675 cells over 4000 MCS while mean cell surface rises
from 22.3 to 44.1 px — a near-doubling that signals the rough, extended invasive
front rather than a smooth compact disc.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `invasion` | `pbg_compucell3d.processes` | 0 | dim_x=150, dim_y=150, total_mcs=4000 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**Composite `pbg_compucell3d.processes`** — `spec_pbg_compucell3d_processes` (materialized from its `@composite_generator`)


In [ ]:
spec_pbg_compucell3d_processes = materialize_composite('pbg_compucell3d.processes')
describe_spec(spec_pbg_compucell3d_processes)

In [ ]:
# === Edit parameters for this composite, then run the Run cell ===
# ov_pbg_compucell3d_processes maps each tunable parameter to its current value (the generator's
# defaults). Edit any entry; the Run cell rebuilds the composite with it.
ov_pbg_compucell3d_processes = dict(spec_pbg_compucell3d_processes['_overrides'])
import json as _json; print(_json.dumps(ov_pbg_compucell3d_processes, indent=2, default=str))
# e.g.:  ov_pbg_compucell3d_processes['<parameter>'] = <value>

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: spheroid-invasion ===
STUDY = 'spheroid-invasion'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_invasion = 100
INTERVAL_invasion = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_pbg_compucell3d_processes if 'ov_pbg_compucell3d_processes' in dir() else None  # edits from the panel cell, if any
            spec_pbg_compucell3d_processes = materialize_composite('pbg_compucell3d.processes', _ov)
            _comp = Composite({'state': spec_pbg_compucell3d_processes['state']}, core=core)
            _comp.run(STEPS_invasion)
        print('ran', 'invasion', f'(pbg_compucell3d.processes) for', STEPS_invasion, 'steps')
    except Exception as _e:
        print(f"invasion not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Population, composition & morphometry**


In [ ]:
# Population, composition & morphometry
show_viz(_render_one('charts/', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| Invasive proliferation morphology on the real engine _(demonstration)_ |  |  |
